# Use regex with file I/O

This guide shows you how to combine regular expressions with Python file handling to read files, match patterns line by line, and process log files.

## The problem

You need to search through files for specific patterns, extract information from log files, or process text files using regular expressions.

## Prerequisites

- Familiarity with basic regex concepts and `re.search()`
- Basic knowledge of Python file handling (`open()`, `with` statements)
- Python 3.12 or later

In [ ]:
import re
from pathlib import Path

## Reading and searching a file

The simplest approach is to read the entire file and search it at once. This works well for small to medium files.

In [ ]:
# Create a sample file for demonstration
sample_text = """Server started at 09:00:00
User alice logged in at 09:15:30
Error: connection timeout at 09:20:45
User bob logged in at 09:25:00
Warning: high memory usage at 09:30:15
User alice logged out at 10:00:00
Error: disk space low at 10:15:30
"""

sample_path = Path('sample_log.txt')
sample_path.write_text(sample_text)
print(f'Created {sample_path}')

In [ ]:
def search_file(file_path: Path, pattern: str) -> list[str]:
    """Search a file for all lines matching the given pattern.

    Args:
        file_path: Path to the file to search.
        pattern: The regex pattern to search for.

    Returns:
        A list of matching lines.
    """
    compiled = re.compile(pattern)
    content = file_path.read_text()
    return [line for line in content.splitlines() if compiled.search(line)]


# Find all error lines
errors = search_file(sample_path, r'Error')
print('Error lines:')
for line in errors:
    print(f'  {line}')

## Processing files line by line

For large files, reading the entire file into memory may not be practical. Process the file line by line instead.

In [ ]:
def find_matching_lines(file_path: Path, pattern: str) -> list[tuple[int, str]]:
    """Find all lines in a file that match the given pattern.

    Args:
        file_path: Path to the file to search.
        pattern: The regex pattern to search for.

    Returns:
        A list of tuples containing the line number and matching line.
    """
    compiled = re.compile(pattern)
    results = []
    with open(file_path) as f:
        for line_number, line in enumerate(f, start=1):
            if compiled.search(line):
                results.append((line_number, line.rstrip()))
    return results


# Find all lines mentioning a user
user_lines = find_matching_lines(sample_path, r'User \w+')
print('Lines mentioning users:')
for line_number, line in user_lines:
    print(f'  Line {line_number}: {line}')

## Extracting data from log files

Combine line-by-line processing with named groups to extract structured data from log files.

In [ ]:
LOG_PATTERN = re.compile(
    r'(?P<type>Error|Warning|User \w+)'
    r'.*?at\s+'
    r'(?P<time>\d{2}:\d{2}:\d{2})'
)


def parse_log_file(file_path: Path) -> list[dict[str, str]]:
    """Parse a log file and extract event types and timestamps.

    Args:
        file_path: Path to the log file.

    Returns:
        A list of dictionaries with 'type' and 'time' keys.
    """
    entries = []
    with open(file_path) as f:
        for line in f:
            match = LOG_PATTERN.search(line)
            if match:
                entries.append(match.groupdict())
    return entries


entries = parse_log_file(sample_path)
print('Log entries:')
for entry in entries:
    print(f'  [{entry["time"]}] {entry["type"]}')

## Searching across multiple files

Use `pathlib.Path.glob()` to search for patterns across multiple files.

In [ ]:
# Create a second sample file
sample_text_2 = """Server restarted at 11:00:00
Error: authentication failed at 11:05:00
User charlie logged in at 11:10:00
"""

sample_path_2 = Path('sample_log_2.txt')
sample_path_2.write_text(sample_text_2)


def search_files(directory: Path, glob_pattern: str, regex_pattern: str) -> dict[str, list[str]]:
    """Search multiple files for lines matching a regex pattern.

    Args:
        directory: The directory to search in.
        glob_pattern: The glob pattern for matching file names.
        regex_pattern: The regex pattern to search for within files.

    Returns:
        A dictionary mapping file names to lists of matching lines.
    """
    compiled = re.compile(regex_pattern)
    results = {}
    for file_path in sorted(directory.glob(glob_pattern)):
        matches = []
        with open(file_path) as f:
            for line in f:
                if compiled.search(line):
                    matches.append(line.rstrip())
        if matches:
            results[file_path.name] = matches
    return results


# Search all .txt files for errors
results = search_files(Path('.'), 'sample_log*.txt', r'Error')
for filename, lines in results.items():
    print(f'{filename}:')
    for line in lines:
        print(f'  {line}')

In [ ]:
# Clean up sample files
sample_path.unlink()
sample_path_2.unlink()
print('Sample files removed.')

## Alternative: using `re.MULTILINE` for whole-file searching

If you read the entire file into a string, use the `re.MULTILINE` flag to make `^` and `$` match the start and end of each line, rather than the start and end of the entire string.

In [ ]:
text = """Line 1: normal
Line 2: Error found
Line 3: normal
Line 4: Error again"""

# Without MULTILINE: ^ only matches start of string
print('Without MULTILINE:', re.findall(r'^.*Error.*$', text))

# With MULTILINE: ^ matches start of each line
print('With MULTILINE:   ', re.findall(r'^.*Error.*$', text, re.MULTILINE))

## Summary

- Read small files entirely with `Path.read_text()` and search with `re.findall()` or `re.finditer()`
- Process large files line by line to conserve memory
- Use named groups with `re.finditer()` to extract structured data
- Use `pathlib.Path.glob()` to search across multiple files
- Use `re.MULTILINE` when searching whole-file content with line-oriented patterns